# 03 — Model Comparison: Physics vs. ML
Head-to-head benchmarking of `PhysicsEstimator` (analytical) vs. `NNEstimator` (1D-CNN).
Uses the trained checkpoint from `run_pipeline.py`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.utils.config import Config
from src.physics.ball_physics_model import BallPhysicsModel
from src.sensors.sensor_noise_simulator import SensorNoiseSimulator
from src.models.physics_estimator import PhysicsEstimator
from src.models.nn_estimator import NNEstimator
from src.evaluation.benchmarker import Benchmarker
from src.data.dataset_generator import DatasetGenerator
from src.utils.visualization import plot_benchmark_comparison, plot_parity

config       = Config()
physics_model = BallPhysicsModel()
noise_sim    = SensorNoiseSimulator(seed=0)

# Load trained model
nn_estimator = NNEstimator.load('../outputs/nn_estimator_best.pt')
print('NN checkpoint loaded')

## 1. Benchmark on 100 test trajectories

In [ ]:
rng = np.random.default_rng(99)
test_traj = [
    physics_model.simulate(
        rng.uniform(5, 35),
        rng.uniform(5, 60),
        rng.uniform(-3000, 3000),
    )
    for _ in range(100)
]

benchmarker = Benchmarker(PhysicsEstimator(), nn_estimator, noise_sim)
results_df  = benchmarker.evaluate(test_traj)
results_df

## 2. MAE comparison bar chart

In [ ]:
Path('../outputs/plots').mkdir(parents=True, exist_ok=True)
plot_benchmark_comparison(results_df, save_path=Path('../outputs/plots/benchmark_comparison.png'))

from IPython.display import Image
Image('../outputs/plots/benchmark_comparison.png', width=800)

## 3. Parity plots — NN predictions vs. true values

In [ ]:
X = np.load('../outputs/dataset_X.npy')
y = np.load('../outputs/dataset_y.npy')

gen = DatasetGenerator(physics_model, SensorNoiseSimulator(42))
_, _, X_test, _, _, y_test = gen.split(X, y)

y_test_phys = gen.denormalise(y_test)
nn_preds    = nn_estimator.predict(X_test[:500])

plot_parity(y_test_phys[:500], nn_preds, save_path=Path('../outputs/plots/parity_plots.png'))
Image('../outputs/plots/parity_plots.png', width=800)

## 4. Key insight: when does each estimator win?

| Metric | PhysicsEstimator | NNEstimator |
|--------|-----------------|-------------|
| Speed MAE (m/s) | ~3.5 | **~0.6** |
| Angle MAE (°) | ~20 | **~1.3** |
| Spin MAE (rpm) | **~0.05** | ~79 |

**Key finding:** The physics estimator excels at estimating **spin** from the gyroscope signal (near-perfect because gyro directly measures spin). The NN dominates for **speed and angle** because it learns the non-linear mapping between the full IMU time-series and launch parameters.

This is the core insight: neither approach is universally better. A hybrid system — using physics for direct observables (spin) and ML for complex inversion (speed, angle) — would outperform either alone.

## 5. Error vs. noise level

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
metrics = [('speed_mae', 'Speed MAE (m/s)'), ('angle_mae', 'Angle MAE (°)')]
colors  = {'PhysicsEstimator': '#4e79a7', 'NNEstimator': '#e15759'}

for ax, (metric, ylabel) in zip(axes, metrics):
    for est in results_df['estimator'].unique():
        subset = results_df[results_df['estimator'] == est]
        ax.plot(subset['noise_level'], subset[metric], 'o-',
                color=colors.get(est, 'grey'), label=est, lw=2)
    ax.set_xlabel('Noise level')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Error vs. Noise Level per Estimator', fontsize=12)
plt.tight_layout()
plt.show()